# To Do in advance

- 1. In the top right corner of this window, you can change the Runtime Type. Switch to T4 (GPU) to enable GPU acceleration and run the analysis much faster. Without GPU acceleration, the analysis will take considerably longer to complete.

- 2. Upload your preprocessed phenotype file and the genotype zip file, which I have uploaded on the Shared folder, into google drive

- 3. Sometimes, after running “pip install ...” an error message appears and you are asked to restart the session. Simply restart the session and run it again, and it will work.

In [ ]:
# In Colab - This is just in case I have commit new changes and torchLIMIX is not up to date
# Makes sure you install the latest version
!pip uninstall torchlimix -y

# Clear pip cache to force fresh download
!pip cache purge

In [ ]:
# Install the package from the github private 
# Sometimes, after running “pip install ...” an error message appears and you are asked to restart the session. Simply restart the session and run it again, and it will work.
!pip install git+https://github.com/bi-horn/torchLIMIX.git

In [ ]:
import os
from google.colab import drive

# Print current working directory
current_path = os.getcwd()
print("Current working directory:", current_path)

In [ ]:
# Upload your data to a google drive folder and adjust the folder name "torchlimix_example_data" below
drive.mount('/content/drive')

# Check your data is there
data_path = '/content/drive/MyDrive/torchlimix_example_data/'
print("Data files:")
for file in os.listdir(data_path):
    print(f"  {file}")


In [ ]:
# Extract to temporary session storage
# Note: Genotype data filtered at MAF 10% can be downloaded with the suffix "_processed".
# The unfiltered data files are named "genotype" or "genotype_1001".
zip_file = "/content/drive/MyDrive/torchlimix_example_data/genotype_horton.zip"
temp_extract = "/content/temp_data/"  # This will be deleted when session ends

!mkdir -p {temp_extract}
!unzip "{zip_file}" -d "{temp_extract}"

print(f"Files extracted to temporary location: {temp_extract}")
print("Note: Files will be deleted when Colab session ends")

In [ ]:
# Here you can see all argument options you have (the relevant ones are already listed in the next cell)
!torchlimix --help

## Hypothesis Tests (`--test_type`)

Multi-trait LMM tests for genetic effects across P environments.

| Test Type | Null Hypothesis (H₀) | Alternative (H₁) | `--pheno_idx` |
|-----------|---------------------|------------------|-------------|
| `common` | No shared genetic effect | β_common ≠ 0 | Not needed |
| `any` | No genetic effect in any environment | At least one β_j ≠ 0 | Not needed |
| `specific` | No effect specific to environment j | β_specific[j] ≠ 0 | **Required** |
| `any_vs_common` | All effects are common (no GxE) | At least one specific effect exists | Not needed |
| `specific_vs_common` | Effect in environment j equals common | β[j] ≠ β_common | **Required** |

### Detailed Descriptions

**`common`**: Tests for persistent/shared genetic effects across all environments

**`any`**: Omnibus test for any genetic association.
- You do not have prior knowledge about the effect and want to test all environments at once

**`specific`**: Tests for environment-specific effect in a single environment.
- Requires `--pheno_idx` (0-indexed): which environment to test.
- Example: `--test_type specific --pheno_idx 2` tests environment 3.

**`any_vs_common`**: Tests for GxE interaction (any environment).
- 3 Hypothesis tests in total: $H_{10}$ tests common effect vs. null (no effect), $H_{20}$ tests any vs. null and $H_{21}$ tests the interction effect (any vs. common)
- Detects if effects differ across environments beyond a shared component.

**`specific_vs_common`**: Tests if one environment deviates from the common effect.
- Requires `--pheno_idx`: which environment to compare against common
- 3 Hypothesis tests in total: $H_{10}$ tests common effect vs. null (no effect), $H_{20}$ tests specific vs. null and $H_{21}$ tests the interction effect (specific vs. common)
- Useful for identifying environment-specific GxE for a particular condition

In [ ]:
# Configuration
# Ignore the --config option (torchLIMIX uses an internal one and you can only overwrite the relevant arguments described in the following)
transformation_method = 'int'     # None since data is already standardized. Otherwise choose between "int" (inverse normal transform) or "z_score"
dataset = "thaliana"              # Name of you dataset
analysis = "gwas"                  # Choose between gwas, vardec (variance decomposition) or prediciton
test_type = "any_vs_common"       # Options: common effect, any effect, specific effect (here you also have to specify pheno_idx), any_vs_common (tests for interaction for all environments at the same time (pv21))
# If your test type is specific_vs_common do not foget to set pheno_idx correctly
pheno_idx = 0                       # Only relevant for specific and specific_vs_common effect test (here you test if the effect is spefific to a certain environment/trait 0<=idx<=(number of traits-1))
rank = 3                          # Model rank parameter: Choose number of phenotypes/environments for best performance (If you choose an invalid parameter it is automatically set to full rank by default)
#device = 1

# File paths
pheno_path = "/content/drive/MyDrive/torchlimix_example_data/thaliana_simulated_phenotype.csv"
geno_path = "/content/temp_data/genotype_processed"
# You can also add annot_path if you already have one otherwise torchLIMIX will create one
output_dir = "/content/drive/MyDrive/torchlimix_results"
root_dir = output_dir # root is path to save the annotation file and cached files if applicable (I save it together with results here)

# Run analysis
!torchlimix \
    --pheno_path {pheno_path} \
    --geno_path {geno_path} \
    --output_directory {output_dir} \
    --dset {dataset} \
    --analysis {analysis} \
    --test_type {test_type} \
    --rank {rank}

# Plot results

In [ ]:
from statsmodels.stats.multitest import multipletests as mt
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import json
from torchlimix.plot._manhattan import manhattan
from torchlimix.plot._qqplot import qqplot

In [ ]:
# Adjust the path to your created annotation file (automatically saved in root directory (default: ../torchLIMIX_output))
annot_file_name = f"{dataset}_snp_annotation.csv"
annot= pd.read_csv(os.path.join(root_dir, annot_file_name), sep=',', index_col=0)
annot.columns = ['chrom', 'pos']
print(annot)

### Log-Likelihood Components

- lml0: Log marginal likelihood under null model ($H_0$: no genetic effect)
- lml1: Log marginal likelihood under alternative model 1 ($H_1$)
- lml2: Log marginal likelihood under alternative model 2 ($H_2$: only for interaction effects)

### Likelihood Ratio Tests (LRT = 2 * (lml_alt - lml_null)):
- llr10: LRT statistic comparing H1 vs H0
- llr20: LRT statistic comparing H2 vs H0
- llr21: LRT statistic comparing H2 vs H1

### Degrees of Freedom:
- df10: df for H1 vs H0 comparison
- df20: df for H2 vs H0 comparison
- df21: df for H2 vs H1 comparison

In [ ]:
file_path = os.path.join(output_dir, "log_likelihoods.csv")
# Load the data
if os.path.isfile(file_path):
    print(f"Loading log-likelihoods from {file_path}.")
    log_likelihoods = pd.read_csv(file_path)
else:
    raise FileNotFoundError(f"Likelihood file does not exist: {file_path}.")

# Extract relevant columns

# Base log-likelihoods (available for all test types)
lml0 = log_likelihoods['lml0'].values    # Null model: no SNP effect
lml1 = log_likelihoods['lml1'].values    # Alternative model 1

# Base LRT (available for all test types)
llr10 = log_likelihoods['lrt10'].values  # LRT: M1 vs M0
df10 = log_likelihoods['df10'].values    # Degrees of freedom for M1 vs M0

# Extended comparisons (only for nested GxE tests)
interaction_tests = ["any_vs_common", "specific_vs_common"]

if test_type in interaction_tests:
    lml2 = log_likelihoods['lml2'].values    # Full model: common + specific effects
    llr20 = log_likelihoods['lrt20'].values  # LRT: M2 vs M0 (full vs null)
    llr21 = log_likelihoods['lrt21'].values  # LRT: M2 vs M1 (specific effect given common)
    df20 = log_likelihoods['df20'].values    # df for M2 vs M0
    df21 = log_likelihoods['df21'].values    # df for M2 vs M1
else:
    # For 'any', 'common', 'specific': only two-model comparison (M0 vs M1)
    lml2 = None
    llr20 = None
    llr21 = None
    df20 = None
    df21 = None

# Check distribution of LRT values
plt.hist(llr10, bins=50, density=True, alpha=0.7, color='blue', label='LRT 10')
#plt.hist(llr20, bins=50, density=True, alpha=0.7, color='green', label='LRT 20')
#plt.hist(llr21, bins=50, density=True, alpha=0.7, color='yellow', label='LRT 12')
plt.xlabel('Likelihood Ratio Test Statistics', fontsize=15)
plt.ylabel('Density', fontsize=15)

plt.tick_params(axis='both', which='major', labelsize=13)  # Adjust 'labelsize' to your preferred size
plt.legend(fontsize=12)
plt.show()

In [ ]:
chromosomes = annot['chrom']
positions = annot['pos']

# Primary result (all test types)
result_mt_10 = pd.DataFrame({
    'chrom': chromosomes,
    'pos': positions,
    'pv': log_likelihoods['pv10'].values
})

# Interaction test results (interaction tests only)
if test_type in interaction_tests:
    result_mt_20 = pd.DataFrame({
        'chrom': chromosomes,
        'pos': positions,
        'pv': log_likelihoods['pv20'].values,
    })
    result_mt_21 = pd.DataFrame({
        'chrom': chromosomes,
        'pos': positions,
        'pv': log_likelihoods['pv21'].values
    })
else:
    result_mt_20 = None
    result_mt_21 = None


In [ ]:
# Select a threshold for significant or suggestive SNPs
# Uncomment the thr

# Option 1: Bonferroni correction (strict) -> alphacBonf_10
# Controls family-wise error rate. Use for identifying high-confidence
# associations, e.g. for follow-up experiments or candidate gene validation.
reject_10, pvals_corrected_10, alphacSidak_10, alphacBonf_10 = mt(
    log_likelihoods['pv10'].values, alpha=0.05, method='bonferroni'
)

# Option 2: Suggestive threshold (lenient) -> e.g. 10**(-4.8)
# A fixed -log10(p) = 4.8 cutoff, useful for exploratory analyses,
# detecting weaker signals, or when sample size limits power.

threshold = alphacBonf_10  # alphacBonf_10 or 10**(-4.8);  uncomment the one you need

In [ ]:
# Manhattan Plot (test-type dependent)

INTERACTION_TESTS = ["any_vs_common", "specific_vs_common"]

fig, ax = plt.subplots(figsize=(12, 6))

if test_type in interaction_tests:
    # Interaction tests: 3 comparisons (H0 vs H1 vs H2)

    # Plot 1: Common effect vs No effect (H1 vs H0)
    manhattan(result_mt_10,
                   colora="#848081",
                   colorb="#848081",
                   highlight_color="#848081",
                   threshold=threshold,
                   pts_kws={'marker': 'o', 'markersize': 7},
                   ax=ax)

    # Plot 2: Full model vs No effect (H2 vs H0)
    manhattan(result_mt_20,
                   colora="#848081",
                   colorb="#848081",
                   highlight_color="#848081",
                   threshold=threshold,
                   pts_kws={'marker': '*', 'markersize': 7},
                   ax=ax)

    # Plot 3: Full model vs Common (H2 vs H1) - GxE interaction test
    manhattan(result_mt_21,
                   colora="#848081",
                   colorb="#848081",
                   highlight_color="#848081",
                   threshold=threshold,
                   pts_kws={'marker': 's', 'markersize': 7},
                   ax=ax)

    # Legend for interaction tests
    handles = [
        plt.Line2D([0], [0], marker='o', color='w', label='Common vs. Null',
                   markerfacecolor='#848081', markersize=7),
        plt.Line2D([0], [0], marker='*', color='w', label='Full vs. Null',
                   markerfacecolor='#848081', markersize=10),
        plt.Line2D([0], [0], marker='s', color='w', label='GxE (Full vs. Common)',
                   markerfacecolor='#848081', markersize=7)
    ]
    title = "GxE Interaction Test" if test_type == "any_vs_common" else f"Specific vs. Common (env {pheno_idx})"

else:
    # Simple tests: single comparison (H1 vs H0)

    # Define color and label based on test type
    plot_config = {
        'common': {'color': '#848081', 'label': 'Common Effect'},
        'any': {'color': '#848081', 'label': 'Any Effect'},
        'specific': {'color': '#848081', 'label': f'Specific Effect (env {pheno_idx})'}
    }

    config = plot_config.get(test_type, {'color': '#5689AC', 'label': test_type})

    manhattan(result_mt_10,
                   colora=config['color'],
                   colorb=config['color'],
                   highlight_color=config['color'],
                   threshold=threshold,
                   pts_kws={'marker': 'o', 'markersize': 7},
                   ax=ax)

    # Legend for simple tests
    handles = [
        plt.Line2D([0], [0], marker='o', color='w', label=config['label'],
                   markerfacecolor=config['color'], markersize=7)
    ]
    title = f"{config['label']} Test"


ax.set_xlabel("Chromosome", fontsize=16)
ax.set_ylabel("-log$_{10}$(p-value)", fontsize=16)

# Compute max -log10(p) from the actual data
if test_type in interaction_tests:
    max_logp = max(
        -np.log10(result_mt_10['pv'].min()),
        -np.log10(result_mt_20['pv'].min()),
        -np.log10(result_mt_21['pv'].min())
    )
else:
    max_logp = -np.log10(result_mt_10['pv'].min())

threshold_log = -np.log10(threshold)
ymax = max(max_logp, threshold_log) * 1.1

ax.set_ylim(0, ymax)

ax.set_title(title, fontsize=18)
ax.tick_params(axis='both', which='major', labelsize=14)
ax.legend(handles=handles, loc='upper right', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# QQ Plot (test-type dependent)
interaction_tests = ["any_vs_common", "specific_vs_common"]

if test_type in interaction_tests:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # QQ Plot 1: Common effect vs Null (H1 vs H0)
    qqplot(log_likelihoods['pv10'].values,
           line=True,
           ax=axes[0],
           label='Common vs. Null',
           alpha=alphacBonf_10,
           pts_kws=dict(marker='o', color='#848081'),
           lambda_in_label=True,
           show_lambda=False)
    axes[0].set_title('Common vs. Null', fontsize=14)
    axes[0].legend(fontsize=10, loc='upper left')

    # QQ Plot 2: Full model vs Null (H2 vs H0)
    qqplot(log_likelihoods['pv20'].values,
           line=True,
           ax=axes[1],
           label='Full vs. Null',
           alpha=alphacBonf_20,
           pts_kws=dict(marker='*', color='#848081'),
           lambda_in_label=True,
           show_lambda=False)
    axes[1].set_title('Full vs. Null', fontsize=14)
    axes[1].legend(fontsize=10, loc='upper left')

    # QQ Plot 3: GxE interaction (H2 vs H1)
    qqplot(log_likelihoods['pv21'].values,
           line=True,
           ax=axes[2],
           label='GxE (Full vs. Common)',
           alpha=alphacBonf_21,
           pts_kws=dict(marker='s', color='#848081'),
           lambda_in_label=True,
           show_lambda=False)
    axes[2].set_title('GxE (Full vs. Common)', fontsize=14)
    axes[2].legend(fontsize=10, loc='upper left')

    # Overall title
    if test_type == "any_vs_common":
        fig.suptitle('QQ Plots: Any vs. Common Test', fontsize=16, y=1.02)
    else:
        fig.suptitle(f'QQ Plots: Specific vs. Common Test (env {pheno_idx})', fontsize=16, y=1.02)

else:
    # Simple tests: single QQ plot
    fig, ax = plt.subplots(figsize=(8, 8))

    plot_config = {
        'common': {'color': '#848081', 'label': 'Common Effect', 'title': 'Common Effect Test'},
        'any': {'color': '#848081', 'label': 'Any Effect', 'title': 'Any Effect Test'},
        'specific': {'color': '#848081', 'label': f'Specific Effect (env {pheno_idx})',
                     'title': f'Specific Effect Test (env {pheno_idx})'}
    }
    config = plot_config.get(test_type, {'color': '#5689AC', 'label': test_type, 'title': test_type})

    qqplot(log_likelihoods['pv10'].values,
           line=True,
           ax=ax,
           label=config['label'],
           alpha=alphacBonf_10,
           pts_kws=dict(marker='o', color=config['color']),
           lambda_in_label=True,
           show_lambda=False)
    ax.set_title(config['title'], fontsize=16)
    ax.legend(fontsize=12, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
interaction_test = ["any_vs_common", "specific_vs_common"]

def print_significant_snps(indices, effect_name, annot):
    """Print significant SNPs with chromosome and position."""
    print(f"\n{effect_name}:")
    print(f"  Total significant: {len(indices)}")
    if len(indices) == 0:
        print("  No significant SNPs")
        return
    sig_snps = annot.iloc[indices][['chrom', 'pos']].copy()
    sig_snps.index.name = 'SNP_index'
    print(sig_snps.to_string())

label_config = {
    'common': 'Common effect',
    'any': 'Any effect',
    'specific': f'Specific effect (env {pheno_idx})',
    'any_vs_common': 'Common effect (H1 vs H0)',
    'specific_vs_common': 'Common effect (H1 vs H0)'
}
label_10 = label_config.get(test_type, 'Primary test')

# Use the same threshold as in the Manhattan plot
significant_indices_10 = np.where(log_likelihoods['pv10'].values < threshold)[0]

print("=" * 60)
print(f"SIGNIFICANT SNPs SUMMARY - Test type: {test_type}")
print(f"Threshold: -log10(p) = {-np.log10(threshold):.2f} (p = {threshold:.2e})")
print("=" * 60)
print_significant_snps(significant_indices_10, f"{label_10} (pv10)", annot)

if test_type in interaction_tests:
    significant_indices_20 = np.where(log_likelihoods['pv20'].values < threshold)[0]
    print_significant_snps(significant_indices_20, "Full model vs Null (pv20)", annot)

    significant_indices_21 = np.where(log_likelihoods['pv21'].values < threshold)[0]
    if test_type == "any_vs_common":
        label_21 = "GxE interaction (any environment)"
    else:
        label_21 = f"GxE interaction (env {pheno_idx} vs common)"
    print_significant_snps(significant_indices_21, f"{label_21} (pv21)", annot)
else:
    significant_indices_20 = None
    significant_indices_21 = None

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"{'Comparison':<35} {'Significant SNPs':<15} {'Threshold':<15}")
print("-" * 60)
print(f"{label_10:<35} {len(significant_indices_10):<15} {threshold:.2e}")
if test_type in interaction_tests:
    print(f"{'Full vs Null':<35} {len(significant_indices_20):<15} {threshold:.2e}")
    print(f"{label_21:<35} {len(significant_indices_21):<15} {threshold:.2e}")
print("=" * 60)
